# Message Utilities

The `utils.py` module provides utilities for serializing, deserializing, converting, filtering, merging, trimming, and approximately counting tokens in LangChain messages.

It also converts LangChain messages into OpenAI-compatible message dictionaries and allows selected message-transformation functions to be used directly or as Runnables.

## Type Aliases

1. `AnyMessage`: Represents any standard LangChain message or message-chunk type.

   It uses a Pydantic discriminator based on the message's `type` field.

   * **Definition:**
     ```python
     AnyMessage = Annotated[
         Annotated[AIMessage, Tag(tag="ai")]
         | Annotated[HumanMessage, Tag(tag="human")]
         | Annotated[ChatMessage, Tag(tag="chat")]
         | Annotated[SystemMessage, Tag(tag="system")]
         | Annotated[FunctionMessage, Tag(tag="function")]
         | Annotated[ToolMessage, Tag(tag="tool")]
         | Annotated[AIMessageChunk, Tag(tag="AIMessageChunk")]
         | Annotated[HumanMessageChunk, Tag(tag="HumanMessageChunk")]
         | Annotated[ChatMessageChunk, Tag(tag="ChatMessageChunk")]
         | Annotated[SystemMessageChunk, Tag(tag="SystemMessageChunk")]
         | Annotated[FunctionMessageChunk, Tag(tag="FunctionMessageChunk")]
         | Annotated[ToolMessageChunk, Tag(tag="ToolMessageChunk")],
         Field(discriminator=Discriminator(_get_type))
     ]
     ```

2. `MessageLikeRepresentation`: Represents the supported forms from which a LangChain message can be created.

   A message may be supplied as an existing `BaseMessage`, a string, a role-content tuple, a dictionary, or a list of strings.

   * **Definition:**
     ```python
     MessageLikeRepresentation = (
         BaseMessage
         | list[str]
         | tuple[
             str,
             str | list[str | dict[str, Any]]
         ]
         | str
         | dict[str, Any]
     )
     ```

## Runnable Support

The following functions can be used in two ways:

- When `messages` is supplied, the function immediately returns the transformed messages.
- When `messages` is omitted or set to `None`, the function returns a `RunnableLambda`.

The supported functions are:

- `filter_messages`
- `merge_message_runs`
- `trim_messages`

### Functions

1. `get_buffer_string`: Converts a sequence of messages into strings and joins them into one buffer.

   The `prefix` format produces `Role: content` output. The `xml` format produces escaped XML-style message elements and supports selected multimodal and tool-call content blocks.

   In XML format, base64 data and unknown content blocks are skipped. Plaintext document content, server-tool arguments, and server-tool results are truncated to 500 characters.

   * **Syntax:**
     ```python
     get_buffer_string(
         messages: Sequence[BaseMessage], # Messages to convert into a string
         human_prefix: str = "Human", # Prefix used for HumanMessage
         ai_prefix: str = "AI", # Prefix used for AIMessage
         *,
         system_prefix: str = "System", # Prefix used for SystemMessage
         function_prefix: str = "Function", # Prefix used for FunctionMessage
         tool_prefix: str = "Tool", # Prefix used for ToolMessage
         message_separator: str = "\n", # Separator inserted between messages
         format: Literal["prefix", "xml"] = "prefix" # Output representation
     ) -> str
     ```

2. `messages_from_dict`: Converts serialized message dictionaries into LangChain message objects.

   Each dictionary must contain the message `type` and its serialized `data`.

   * **Syntax:**
     ```python
     messages_from_dict(
         messages: Sequence[dict[str, Any]] # Serialized message dictionaries
     ) -> list[BaseMessage]
     ```

3. `message_chunk_to_message`: Converts a message chunk into its corresponding non-chunk message class.

   If the supplied object is already a normal message, it is returned unchanged. Chunk-only fields such as `tool_call_chunks` and `chunk_position` are removed from AI message chunks.

   * **Syntax:**
     ```python
     message_chunk_to_message(
         chunk: BaseMessage # Message or message chunk to convert
     ) -> BaseMessage
     ```

4. `convert_to_messages`: Converts message-like representations or a `PromptValue` into a list of `BaseMessage` objects.

   Supported representations include existing messages, strings, role-content tuples, dictionaries, and supported serialized constructor envelopes.

   * **Syntax:**
     ```python
     convert_to_messages(
         messages: Iterable[MessageLikeRepresentation]
         | PromptValue # Message representations or prompt value
     ) -> list[BaseMessage]
     ```

5. `filter_messages`: Filters messages using their names, types, identifiers, or tool-call identifiers.

   A message must satisfy at least one supplied inclusion condition and must not satisfy any exclusion condition. When no inclusion condition is supplied, every message that is not explicitly excluded is retained.

   When `exclude_tool_calls=True`, AI messages containing tool calls and all tool-result messages are removed. A sequence of tool-call IDs removes only matching tool calls and their corresponding tool messages.

   This function can also return a Runnable when `messages` is omitted.

   * **Syntax:**
     ```python
     filter_messages(
         messages: Iterable[MessageLikeRepresentation]
         | PromptValue, # Messages to filter
         *,
         include_names: Sequence[str] | None = None, # Message names to include
         exclude_names: Sequence[str] | None = None, # Message names to exclude
         include_types: Sequence[
             str | type[BaseMessage]
         ] | None = None, # Message types to include
         exclude_types: Sequence[
             str | type[BaseMessage]
         ] | None = None, # Message types to exclude
         include_ids: Sequence[str] | None = None, # Message IDs to include
         exclude_ids: Sequence[str] | None = None, # Message IDs to exclude
         exclude_tool_calls: Sequence[str]
         | bool
         | None = None # Tool-call IDs or all tool calls to exclude
     ) -> list[BaseMessage]
     ```

6. `merge_message_runs`: Merges consecutive messages that have the same message class.

   String contents are joined using `chunk_separator`. When at least one message has list-based content, the resulting content remains a list of content blocks.

   `ToolMessage` objects are never merged because each tool result has a distinct tool-call identifier.

   This function can also return a Runnable when `messages` is omitted.

   * **Syntax:**
     ```python
     merge_message_runs(
         messages: Iterable[MessageLikeRepresentation]
         | PromptValue, # Consecutive messages to merge
         *,
         chunk_separator: str = "\n" # Separator between string contents
     ) -> list[BaseMessage]
     ```

7. `trim_messages`: Trims a message history so that it stays within a maximum token or message count.

   The `first` strategy retains the beginning of the history, while the `last` strategy retains the most recent messages. Partial message content may be retained when `allow_partial=True`.

   A language model, a custom counting function, `len`, or the `"approximate"` shortcut can be used as the token counter. The `"approximate"` shortcut uses `count_tokens_approximately`.

   `start_on` and `include_system` are intended for the `last` strategy. `end_on` restricts the final message type, and `text_splitter` controls how partially retained text is split.

   This function can also return a Runnable when `messages` is omitted.

   * **Syntax:**
     ```python
     trim_messages(
         messages: Iterable[MessageLikeRepresentation]
         | PromptValue, # Message history to trim
         *,
         max_tokens: int, # Maximum allowed token or message count
         token_counter: Callable[
             [list[BaseMessage]],
             int
         ]
         | Callable[
             [BaseMessage],
             int
         ]
         | BaseLanguageModel[Any]
         | Literal["approximate"], # Token-counting strategy
         strategy: Literal[
             "first",
             "last"
         ] = "last", # Side of the history to preserve
         allow_partial: bool = False, # Whether partial messages may be retained
         end_on: str
         | type[BaseMessage]
         | Sequence[
             str | type[BaseMessage]
         ]
         | None = None, # Message type on which the result should end
         start_on: str
         | type[BaseMessage]
         | Sequence[
             str | type[BaseMessage]
         ]
         | None = None, # Message type on which the result should start
         include_system: bool = False, # Preserve the initial SystemMessage
         text_splitter: Callable[
             [str],
             list[str]
         ]
         | TextSplitter
         | None = None # Splitter used for partial text content
     ) -> list[BaseMessage]
     ```

8. `convert_to_openai_messages`: Converts LangChain messages into OpenAI-compatible message dictionaries.

   It accepts a single message-like object or a sequence. A single input returns one dictionary, while a sequence returns a list of dictionaries.

   The function translates supported OpenAI, Anthropic, Bedrock Converse, and Vertex AI content-block formats. Unknown blocks are either passed through or rejected according to `pass_through_unknown_blocks`.

   * **Syntax:**
     ```python
     convert_to_openai_messages(
         messages: MessageLikeRepresentation
         | Sequence[
             MessageLikeRepresentation
         ], # Message or messages to convert
         *,
         text_format: Literal[
             "string",
             "block"
         ] = "string", # Representation used for text content
         include_id: bool = False, # Include available message IDs
         pass_through_unknown_blocks: bool = True # Preserve unknown content blocks
     ) -> dict[str, Any] | list[dict[str, Any]]
     ```

9. `count_tokens_approximately`: Estimates the total token count of messages and optional tool schemas.

   The estimate includes message content, role names, optional message names, AI tool calls, and tool-call identifiers. Images use a fixed token cost instead of counting base64 characters.

   When `use_usage_metadata_scaling=True`, the estimate may be scaled using the most recent AI usage metadata when all AI messages use a consistent model provider.

   This is an approximation and may differ from model-specific tokenization.

   * **Syntax:**
     ```python
     count_tokens_approximately(
         messages: Iterable[
             MessageLikeRepresentation
         ], # Messages whose tokens should be estimated
         *,
         chars_per_token: float = 4.0, # Approximate characters per token
         extra_tokens_per_message: float = 3.0, # Additional tokens per message
         count_name: bool = True, # Include message names in the estimate
         tokens_per_image: int = 85, # Fixed token cost assigned to each image
         use_usage_metadata_scaling: bool = False, # Scale using AI usage metadata
         tools: list[
             BaseTool | dict[str, Any]
         ] | None = None # Tool schemas to include in the count
     ) -> int
     ```